## Data Loading

In [44]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
from sklearn import preprocessing
import numpy as np
import random

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from transformers import BertTokenizer, BertModel
import torch

# load_aokvqa.py
import os
import json

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def load_aokvqa(aokvqa_dir, split, version='v1p0'):
    assert split in ['train', 'val', 'test', 'test_w_ans']
    dataset = json.load(open(
        os.path.join(aokvqa_dir, f"aokvqa_{version}_{split}.json")
    ))
    return dataset

def get_coco_path(split, image_id, coco_dir):
    return os.path.join(coco_dir, f"{split}2017", f"{image_id:012}.jpg")

AOKVQA_DIR="datasets/aokvqa/"
COCO_DIR="datasets/coco/"
aokvqa_dir = f"./aokvqa/{AOKVQA_DIR}"
coco_dir = f"./aokvqa/{COCO_DIR}"

Using device: cuda


In [45]:
def subsample(dataset, n_samples=None, frac=None, random_seed=42):
    random.seed(random_seed)
    if n_samples:
        return random.sample(dataset, n_samples)
    elif frac:
        sample_size = int(len(dataset) * frac)
        return random.sample(dataset, sample_size)
    return dataset

In [46]:
USE_SUBSET_DATA = False 
train_dataset = load_aokvqa(aokvqa_dir, 'train')  
val_dataset = load_aokvqa(aokvqa_dir, 'val')
test_dataset = load_aokvqa(aokvqa_dir, 'test')
print(f"Full Train aokvqa: {len(train_dataset)}")
print(f"Full Val aokvqa: {len(val_dataset)}")
print(f"Full Test aokvqa: {len(test_dataset)}")

if USE_SUBSET_DATA:
    train_dataset = subsample(train_dataset, frac=0.2)
    val_dataset = subsample(val_dataset, frac=0.2)
    test_dataset = subsample(test_dataset, frac=0.2)
    print(f"Used Train aokvqa: {len(train_dataset)}")
    print(f"Used Val aokvqa: {len(val_dataset)}")
    print(f"Used Test aokvqa: {len(test_dataset)}")

Full Train aokvqa: 17056
Full Val aokvqa: 1145
Full Test aokvqa: 6702


## Data Preparation

Fields Considered:

- Question
- Choices
- Correct answer
- Correct Choice Indice
- Rationale
- Direct answer

In [47]:
qa_data = []
for sample in val_dataset:
    question_id = sample["question_id"]
    image_id = sample["image_id"]
    question = sample["question"]
    choices = sample["choices"]
    correct_choice_idx = sample["correct_choice_idx"]
    rationales = sample.get("rationales", [])
    combined_rationale = " ".join(rationales)
    direct_answers = sample.get("direct_answers", [])
    combined_direct_answer = " ".join(direct_answers)
    correct_ans = choices[correct_choice_idx]
    qa_data.append({
        "question_id": question_id,
        "image_id": image_id,
        "question": question,
        "choices": choices,
        "correct_answer": correct_ans,
        "correct_choice_idx": correct_choice_idx,
        "rationale": combined_rationale,
        "direct_answer": combined_direct_answer
    })
qa_df = pd.DataFrame(qa_data)

In [48]:
qa_df.head()


,question_id,image_id,question,choices,correct_answer,correct_choice_idx,rationale,direct_answer
0,22jbM6gDxdaMaunuzgrsBB,461751,What is in the motorcyclist's mouth?,"[toothpick, food, popsicle stick, cigarette]",cigarette,3,He's smoking while riding. The motorcyclist ha...,cigarette cigarette cigarette cigarette cigare...
1,2Aq5RiEn7eyfWjEbpuYT2o,377368,Which number birthday is probably being celebr...,"[one, ten, nine, thirty]",thirty,3,There is a birthday cake on the table with the...,thirty 30th thirty thirty thirty 30th thirty t...
2,2Br4bJfKY7SQM9DECrqqeG,563603,What best describes the pool of water?,"[frozen, fresh, dirty, boiling]",dirty,2,The pool is dark brown. It it brown and surrou...,muddy dirty murky water muddy pond pond wateri...
3,2C8riXpRLX3CyM5jDz23m7,329542,What is the white substance on top of the cupc...,"[butter, mayo, ice cream, icing]",icing,3,This is frosting used to decorate and add more...,icing whipped cream icing frosting icing frost...
4,2DQex53EkNGH2cfo3WPuPn,182202,What type of device is sitting next to the lap...,"[mouse, mobile phone, pen, keyboard]",mobile phone,1,It has the name of it on the top The device ha...,cell phone vodafone phone phone phone phone ce...


## Unimodal Baseline (Text)

### Random

### Majority Class Baseline

In [49]:
from collections import Counter

# Count answer frequencies
all_answers = [sample['correct_answer'] for sample in qa_data]
answers_count = Counter(all_answers)

qa_df['majority_class_prediction'] = qa_df['choices'].apply(
    lambda choice_list: max(choice_list, key=lambda c: answers_count[c])
)
accuracy = (qa_df['majority_class_prediction'] == qa_df['correct_answer']).mean()
print(f"Majority Class Baseline Accuracy: {accuracy:.2%}")


Majority Class Baseline Accuracy: 66.20%


## Unimodal Baseline (Image)

### YOLO v8

In [50]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

def get_yolo_prediction(img_path, choices):
    results = model(img_path)
    result = results[0]
    if result.boxes.cls.numel() > 0:
        detected_indices = result.boxes.cls.cpu().numpy()
        detected_classes = [model.names[int(idx)] for idx in detected_indices]
    else:
        detected_classes = []

    for choice in choices:
        if any(choice.lower() in detection.lower() or detection.lower() in choice.lower() for detection in detected_classes):
            return choice
    return choices[0]

predictions = []
for sample in qa_data:
    image_id = sample['image_id']
    img_path = get_coco_path('val', image_id, coco_dir)
    choices  = sample['choices'] 
    prediction = get_yolo_prediction(img_path, choices)
    predictions.append(prediction)

qa_df['yolo_baseline_prediction'] = predictions
accuracy = (qa_df['yolo_baseline_prediction'] == qa_df['correct_answer']).mean()
print(f"YOLO Baseline Accuracy: {accuracy:.2%}")


image 1/1 /home/mscvjj/yuze/MLLM-AOKVQA/aokvqa/datasets/coco/val2017/000000461751.jpg: 576x640 4 persons, 1 car, 1 motorcycle, 6.4ms
Speed: 2.5ms preprocess, 6.4ms inference, 1.1ms postprocess per image at shape (1, 3, 576, 640)

image 1/1 /home/mscvjj/yuze/MLLM-AOKVQA/aokvqa/datasets/coco/val2017/000000377368.jpg: 512x640 1 person, 2 cups, 2 cakes, 2 dining tables, 6.1ms
Speed: 2.2ms preprocess, 6.1ms inference, 0.8ms postprocess per image at shape (1, 3, 512, 640)

image 1/1 /home/mscvjj/yuze/MLLM-AOKVQA/aokvqa/datasets/coco/val2017/000000563603.jpg: 640x480 3 giraffes, 5.8ms
Speed: 2.0ms preprocess, 5.8ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 /home/mscvjj/yuze/MLLM-AOKVQA/aokvqa/datasets/coco/val2017/000000329542.jpg: 640x640 1 person, 4 cakes, 6.3ms
Speed: 5.7ms preprocess, 6.3ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 /home/mscvjj/yuze/MLLM-AOKVQA/aokvqa/datasets/coco/val2017/000000182202.jpg: 416x640 1 lap

YOLO Baseline Accuracy: 27.69%


## Fast RCNN

In [51]:
import torch
import torchvision
from torchvision.transforms import functional as F
from PIL import Image
from tqdm import tqdm

torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
model.to(device)
model.eval()

COCO_INSTANCE_CATEGORY_NAMES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant',
    'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A', 'N/A', 'handbag',
    'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat',
    'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass',
    'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange',
    'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch',
    'potted plant', 'bed', 'N/A', 'dining table', 'N/A', 'N/A', 'toilet', 'N/A', 'tv',
    'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster',
    'sink', 'refrigerator', 'N/A', 'book', 'clock', 'vase', 'scissors', 'teddy bear',
    'hair drier', 'toothbrush'
]

def get_rcnn_prediction(img_path, choices, score_threshold=0.5):
    image = Image.open(img_path).convert("RGB")
    image_tensor = F.to_tensor(image).to(device)
    
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            predictions = model([image_tensor])
    prediction = predictions[0]
    
    if len(prediction["labels"]) > 0:
        keep = prediction["scores"] > score_threshold
        detected_indices = prediction["labels"][keep].cpu().numpy()
        detected_classes = [COCO_INSTANCE_CATEGORY_NAMES[int(idx)] for idx in detected_indices]
    else:
        detected_classes = []
    
    for choice in choices:
        if any(choice.lower() in detection.lower() or detection.lower() in choice.lower() 
               for detection in detected_classes):
            return choice
    return choices[0]

predictions = []
for sample in tqdm(qa_data, desc="Processing Images"):
    image_id = sample['image_id']
    img_path = get_coco_path('val', image_id, coco_dir)
    choices  = sample['choices'] 
    prediction = get_rcnn_prediction(img_path, choices)
    predictions.append(prediction)

qa_df['rcnn_baseline_prediction'] = predictions
accuracy = (qa_df['rcnn_baseline_prediction'] == qa_df['correct_answer']).mean()
print(f"RCNN Baseline Accuracy: {accuracy:.2%}")

Processing Images: 100%|██████████| 1145/1145 [01:22<00:00, 13.91it/s]

RCNN Baseline Accuracy: 27.42%


## Multimodal

### CLIP

In [52]:
import torch
import clip
from PIL import Image
from tqdm import tqdm

# Set up device and load the CLIP model along with its preprocessing pipeline.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()

def get_clip_prediction(img_path, choices):
    # Load and preprocess the image
    image = Image.open(img_path).convert("RGB")
    image_input = preprocess(image).unsqueeze(0).to(device)
    
    # Tokenize the list of candidate text choices
    text_input = clip.tokenize(choices).to(device)
    
    with torch.no_grad():
        # Compute image and text features
        image_features = model.encode_image(image_input)
        text_features = model.encode_text(text_input)
        
        # Normalize features to unit length (recommended for cosine similarity)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        
        # Compute similarity between the image and each text choice
        similarity = (image_features @ text_features.T).squeeze(0)
    
    # Return the choice with the highest similarity score
    best_idx = similarity.argmax().item()
    return choices[best_idx]

# Process each sample in qa_data and compute predictions
predictions = []
for sample in tqdm(qa_data, desc="Processing Images"):
    image_id = sample['image_id']
    img_path = get_coco_path('val', image_id, coco_dir)
    choices  = sample['choices'] 
    prediction = get_clip_prediction(img_path, choices)
    predictions.append(prediction)

qa_df['clip_baseline_prediction'] = predictions
accuracy = (qa_df['clip_baseline_prediction'] == qa_df['correct_answer']).mean()
print(f"CLIP Baseline Accuracy: {accuracy:.2%}")

100%|███████████████████████████████████████| 338M/338M [00:07<00:00, 44.9MiB/s]
Processing Images: 100%|██████████| 1145/1145 [00:19<00:00, 57.70it/s]

CLIP Baseline Accuracy: 56.16%
